# 27D. Text-Image FN-Masked — SigLIP2 Unfrozen-Layer Selection (Mean-Pooled Text)

Adapts **NB 27C** (SigLIP2 layer sweep with chunk-and-max text handling) to use **mean-pooling**, per **NB 09**'s finding that max-over-chunks lets the model ignore most of each text passage and flatters GT metrics.

### Changes from 27C

| Axis | 27C | **27D (this notebook)** |
|---|---|---|
| **Text chunking** | 64-token chunks | same (64 tokens) |
| **Training similarity** | `sim[i,j] = max_k (image_i · chunk_j^k)` | **mean-pool chunks → single vector → `sim[i,j] = image_i · pooled_text_j`** |
| **GT eval** | same max-similarity rule | **same mean-pool rule** (matches training) |
| **Encoder LR** | 1e-6 | 1e-6 (unchanged) |
| **Other settings** | 20 epochs; `n` = `[0, 2, 4, 6, 8]` | 30 epochs; `n` = `[0, 2, 4, 8, 12]`; same encoder, LR, batch, FN mask, and evaluation cadence |
| **Output dir** | `textimage_siglip2_fn_masked` | **`textimage_siglip2_meanpool`** (separate; don't mix with 27C artifacts) |

### Why mean-pooling

Max-similarity only requires **one** chunk to match the image well — the model can ignore the remaining chunks entirely, so long BSV texts effectively reduce to "whichever 64-token fragment looks most image-like." Mean-pooling forces **every** chunk to contribute: the pooled vector is `normalize(mean(project(chunk_k)))`, and the training/eval signal flows through all chunks. NB 09 showed this yields **+20–35% R@1** on a fair GT evaluation versus max-pooling.

### What `n` means here

`n_unfrozen_layers` selects how many of the **last** layers to unfreeze in **both** SigLIP2 towers (12 layers per tower). The current sweep is `[0, 2, 4, 8, 12]`; `n=12` unfreezes the full towers. The historical run requested `n=16` by mistake, which also selected all 12 available layers; its saved artifacts are under `last_16_layers/`.

### GT cadence — every 2 epochs

`TARGET_EPOCHS = 30` and `GT_EVERY_N_EPOCHS = 2` → GT is evaluated at epochs 2, 4, 6, …, 30.

### Controlled-comparison design

Same 2500 records for every `n` (`random.Random(RANDOM_SEED + SIZE)`), identical batch order across `n` (`np.random.seed(RANDOM_SEED)` before each epoch loop).

### Requires NB28's pickles

Loads `vars/raw_textimage.pkl`, `vars/nb28_textimage_records.pkl`, `vars/raw_gt_textimage.pkl`, `vars/gt_textimage_records.pkl`. If missing, run NB28 Section 2 + Section 6 once.

### Output layout

```
OUTPUT_PARENT = /projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/textimage_siglip2_meanpool
  img_inbatch_layers_ft.log
  summary_all_n.csv
  summary_per_epoch_all_n.csv
  last_0_layers/  last_2_layers/  last_4_layers/  last_8_layers/  last_12_layers/
    latest_checkpoint.pt
    training_history.csv
    gt_results.csv
    per_epoch_gt/epoch_02,04,06,...,30_gt_results.csv

The embedded outputs and original checkpoint for the all-layer run retain the historical `last_16_layers/` name; that request unfroze the same 12 layers. The corrected source configuration uses `last_12_layers/`.
    head_text_siglip2.pt
    head_image_siglip2.pt
    siglip2_vision_unfrozen.pt
    siglip2_text_unfrozen.pt
```

### Train-or-resume semantics (matches 27C / NB 02 / NB 09)

| State on disk (per `n`) | Behaviour |
|---|---|
| `last_{n}_layers/gt_results.csv` **exists** | Skip; return existing summary |
| `last_{n}_layers/latest_checkpoint.pt` **exists**, epoch < 30 | Resume |
| `last_{n}_layers/latest_checkpoint.pt` **exists**, epoch ≥ 30 | Finalize only |
| `last_{n}_layers/latest_checkpoint.pt` **missing** | **Train from scratch**, epoch 0 → 30 |

## Section 0 — Installations

Same pin set as 27C / NB 02 / NB 09.

In [1]:
# !pip install openpyxl
# !pip install pandas --upgrade
# !pip install accelerate
# !pip install transformers==4.57.6   # SigLIP2 + get_image_features API
# !pip install --upgrade huggingface_hub

### SigLIP2 is **not** gated — no HuggingFace login required.

## Section 1 — Setup, logging, config

Logging, HF cache, and pickle helpers identical to 27C. **GPU pinned to `cuda:0`**.

In [2]:
import json
import os
import re
import sys
import time
import math
import random
import pickle
import logging
import gc
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModel, AutoProcessor

import transformers

In [3]:
transformers.logging.set_verbosity_error()

OUTPUT_PARENT = "/projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/textimage_siglip2_meanpool"
os.makedirs(OUTPUT_PARENT, exist_ok=True)

LOG_FORMAT = '%(asctime)s : %(levelname)s : %(message)s'
LOG_FILE   = os.path.join(OUTPUT_PARENT, "img_inbatch_layers_ft.log")

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("img_inbatch_ft")
if not any(type(h) is logging.StreamHandler for h in logger.handlers):
    _sh = logging.StreamHandler(sys.stdout)
    _sh.setFormatter(logging.Formatter(LOG_FORMAT))
    logger.addHandler(_sh)


# --- GPU pinning: always cuda:0 ---
CUDA = "cuda:0"
device = CUDA if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)
logger.info("Device: %s", device)
if device == CUDA:
    logger.info("GPU: %s (physical index %d of %d visible)",
                torch.cuda.get_device_name(torch.cuda.current_device()),
                torch.cuda.current_device(),
                torch.cuda.device_count())

VARS_DIR = "vars"
os.makedirs(VARS_DIR, exist_ok=True)

def pkl_vars(varname, filename):
    with open(filename, 'wb') as file:
        pickle.dump(varname, file)

def reload_vars(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

2026-10-22 19:36:47,521 : INFO : Device: cuda:0
2026-10-22 19:36:47,524 : INFO : GPU: Quadro RTX 8000 (physical index 0 of 2 visible)


### Config

- **`SIGLIP2_MODEL_NAME = "google/siglip2-base-patch16-224"`**
- **`SIZE = 2500`**, **`TARGET_EPOCHS = 30`**, **`GT_EVERY_N_EPOCHS = 2`**
- **`N_LAYERS_TO_TRY = [0, 2, 4, 8, 12]`** — `12` unfreezes all layers in both SigLIP2 towers.
- **`SIGLIP2_TEXT_CHUNK_TOKENS = 64`** — same as 27C
- **`SIGLIP2_ENCODER_LR = 1e-6`** — same as 27C (NB 02's conservative value)
- **`HEAD_LEARNING_RATE = 1e-4`**, **`TEMPERATURE = 0.07`**, **`SHARED_DIM = 512`**, **`BATCH_SIZE = 32`** — unchanged

In [4]:
# --- Shared ---
HF_CACHE          = "/projects/agrifutur/hf_cache2"
RANDOM_SEED       = 42
SIZE              = 2500
TARGET_EPOCHS     = 30
GT_EVERY_N_EPOCHS = 2
N_LAYERS_TO_TRY   = [0, 2, 4, 8, 12]            # 0 = fully frozen baseline; 12 = unfreeze all

SIGLIP2_MODEL_NAME        = "google/siglip2-base-patch16-224"
SIGLIP2_TEXT_CHUNK_TOKENS = 64
SIGLIP2_ENCODER_LR        = 1e-6

IMAGE_ENCODE_CHUNK_SIZE = 16

LOG_EVERY_N_STEPS        = 500
CHECKPOINT_EVERY_N_STEPS = 500
MAX_RUNTIME_HOURS_PER_SIZE = 10.0

K_VALUES          = [1, 3, 5, 10]
HEAD_LEARNING_RATE = 1e-4
TEMPERATURE       = 0.07
SHARED_DIM        = 512
BATCH_SIZE        = 32
FN_SIMILARITY_THRESHOLD = 0.8

_MODEL_FILE_RENAMES = {
    "vision_model": "siglip2_vision_unfrozen",
    "text_model":   "siglip2_text_unfrozen",
}

# --- INPUT paths ---
FN_METADATA_PATH    = "/projects/agrifutur/yabusaqe/Finetuning/Shared_data/df_combined_metadata.pkl"
FN_IMAGE_IMAGE_PATH = "/projects/agrifutur/yabusaqe/Finetuning/Shared_data/reranked_image_image.pkl"
IMG_BASE_DIR        = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Automatic Construction"
TEXTIMAGE_PAIRS_PATH = os.path.join(IMG_BASE_DIR, "bsv_pairs_auto_textimage.jsonl")
IMAGE_DIR           = os.path.join(IMG_BASE_DIR, "bsv_images_auto_textimage")
GT_TEXTIMAGE_PATH   = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_gt_textimage_final_500.jsonl"
GT_IMAGE_DIR        = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_images_gt_2404"

logger.info("Config loaded. OUTPUT_PARENT=%s", OUTPUT_PARENT)
logger.info("SIZE=%d, TARGET_EPOCHS=%d, GT_EVERY_N_EPOCHS=%d, N_LAYERS_TO_TRY=%s",
            SIZE, TARGET_EPOCHS, GT_EVERY_N_EPOCHS, N_LAYERS_TO_TRY)
logger.info("Text handling: chunk_size=%d, MEAN-POOL reduction", SIGLIP2_TEXT_CHUNK_TOKENS)
logger.info("LRs: head=%.1e, encoder=%.1e", HEAD_LEARNING_RATE, SIGLIP2_ENCODER_LR)

2026-10-22 19:39:37,013 : INFO : Config loaded. OUTPUT_PARENT=/projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/textimage_siglip2_meanpool
2026-10-22 19:39:37,017 : INFO : SIZE=2500, TARGET_EPOCHS=30, GT_EVERY_N_EPOCHS=2, N_LAYERS_TO_TRY=[0, 2, 4, 8, 16]
2026-10-22 19:39:37,018 : INFO : Text handling: chunk_size=64, MEAN-POOL reduction
2026-10-22 19:39:37,020 : INFO : LRs: head=1.0e-04, encoder=1.0e-06


## Section 2 — Load data (reuse NB28's pickles)

In [5]:
REQUIRED_PICKLES = {
    "raw_textimage":          os.path.join(VARS_DIR, "raw_textimage.pkl"),
    "nb28_textimage_records": os.path.join(VARS_DIR, "nb28_textimage_records.pkl"),
    "raw_gt_textimage":       os.path.join(VARS_DIR, "raw_gt_textimage.pkl"),
    "gt_textimage_records":   os.path.join(VARS_DIR, "gt_textimage_records.pkl"),
}
missing = [p for p in REQUIRED_PICKLES.values() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Missing NB28 pickles:\n  " + "\n  ".join(missing) +
        "\nRun NB28's Section 2 and Section 6 once to build them, then re-run this cell."
    )

raw_textimage        = reload_vars(REQUIRED_PICKLES["raw_textimage"])
textimage_records    = reload_vars(REQUIRED_PICKLES["nb28_textimage_records"])
raw_gt_textimage     = reload_vars(REQUIRED_PICKLES["raw_gt_textimage"])
gt_textimage_records = reload_vars(REQUIRED_PICKLES["gt_textimage_records"])

logger.info("Loaded NB28 pickles: %d training records, %d GT records",
            len(textimage_records), len(gt_textimage_records))
assert len(textimage_records) == 33526
assert len(textimage_records) > SIZE
assert len(gt_textimage_records) > 0

_sample = textimage_records[0]
for _k in ("bulletin_id", "section_title", "text", "image_path"):
    assert _k in _sample, f"missing key {_k!r} in textimage_records"
logger.info("Training record schema OK")

2026-10-22 19:39:51,721 : INFO : Loaded NB28 pickles: 33526 training records, 500 GT records
2026-10-22 19:39:51,724 : INFO : Training record schema OK


## Section 3 — False-negative mask (image-image, verbatim from 27C)

In [6]:
def build_key(bulletin_id, section_title, text):
    return (bulletin_id, section_title, text)

def build_fn_mask_set(pkl_path, threshold=FN_SIMILARITY_THRESHOLD):
    df = pd.read_pickle(pkl_path)
    hi = df[df["colbert"] >= threshold]
    pairs = set()
    for _, row in hi.iterrows():
        pairs.add(frozenset({int(row["global_i"]), int(row["global_j"])}))
    return pairs

fn_metadata = pd.read_pickle(FN_METADATA_PATH)

fn_image_image = build_fn_mask_set(FN_IMAGE_IMAGE_PATH)
key_to_globals_image = defaultdict(list)
for _, row in fn_metadata[fn_metadata["origin"] == "text_image"].iterrows():
    k = build_key(row["bulletin_id"], row["section_title"], row["text"])
    key_to_globals_image[k].append(row["_global"])
logger.info("Image-image: %d flagged pairs, %d unique keys", len(fn_image_image), len(key_to_globals_image))

def is_masked_image_image(record_i, record_j, threshold_set=fn_image_image):
    key_i = build_key(record_i["bulletin_id"], record_i["section_title"], record_i["text"])
    key_j = build_key(record_j["bulletin_id"], record_j["section_title"], record_j["text"])
    for gi in key_to_globals_image.get(key_i, []):
        for gj in key_to_globals_image.get(key_j, []):
            if frozenset({gi, gj}) in threshold_set:
                return True
    return False

logger.info("is_masked_image_image() ready.")

2026-10-22 19:39:59,499 : INFO : Image-image: 240959 flagged pairs, 27476 unique keys
2026-10-22 19:39:59,501 : INFO : is_masked_image_image() ready.


## Section 4 — Model builder (SigLIP2, parameterized by `n_unfrozen_layers`)

Same as 27C — encoder probing, layer unfreezing, optimizer construction. **Text encoding returns chunk embeddings** `(n_chunks, TEXT_DIM)`; the mean-pool reduction happens in the loss/eval functions, not here.

In [7]:
def build_models_imagetext(n_unfrozen_layers):
    assert n_unfrozen_layers >= 0, f"n_unfrozen_layers must be >= 0, got {n_unfrozen_layers}"

    processor = AutoProcessor.from_pretrained(SIGLIP2_MODEL_NAME, cache_dir=HF_CACHE)
    tokenizer = processor.tokenizer

    siglip_model = AutoModel.from_pretrained(SIGLIP2_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    vision_model = siglip_model.vision_model
    text_model   = siglip_model.text_model

    # Freeze all
    for p in siglip_model.parameters():
        p.requires_grad = False

    n_vision_layers = len(vision_model.encoder.layers)
    n_text_layers   = len(text_model.encoder.layers)
    if n_unfrozen_layers > n_vision_layers:
        logger.warning("n=%d > vision layers (%d); unfreezing all", n_unfrozen_layers, n_vision_layers)
    if n_unfrozen_layers > n_text_layers:
        logger.warning("n=%d > text layers (%d); unfreezing all", n_unfrozen_layers, n_text_layers)

    if n_unfrozen_layers > 0:
        for layer in vision_model.encoder.layers[-n_unfrozen_layers:]:
            for p in layer.parameters():
                p.requires_grad = True
        for layer in text_model.encoder.layers[-n_unfrozen_layers:]:
            for p in layer.parameters():
                p.requires_grad = True

    # Probe projected feature dims
    with torch.no_grad():
        _probe_img = Image.new("RGB", (224, 224), color="white")
        _probe_img_inputs = processor(images=[_probe_img], return_tensors="pt").to(device)
        _probe_img_feat = siglip_model.get_image_features(**_probe_img_inputs)
        VISION_DIM = _probe_img_feat.shape[-1]

        _probe_txt_inputs = tokenizer(
            ["probe"], padding="max_length", max_length=SIGLIP2_TEXT_CHUNK_TOKENS,
            truncation=True, return_tensors="pt",
        ).to(device)
        _probe_txt_feat = siglip_model.get_text_features(**_probe_txt_inputs)
        TEXT_DIM = _probe_txt_feat.shape[-1]

    def chunk_text_for_siglip(text, chunk_tokens=SIGLIP2_TEXT_CHUNK_TOKENS):
        """Split text into ~chunk_tokens-token chunks."""
        ids = tokenizer.encode(text, add_special_tokens=False)
        if not ids:
            return [text]
        chunks = [ids[i:i + chunk_tokens] for i in range(0, len(ids), chunk_tokens)]
        return [tokenizer.decode(c) for c in chunks]

    def encode_siglip2_text_chunks(text):
        """Returns chunk embeddings (n_chunks, TEXT_DIM). No reduction here --
        mean-pooling is applied in compute_imagetext_loss / evaluate_imagetext."""
        chunks = chunk_text_for_siglip(text)
        inputs = tokenizer(chunks, padding="max_length", max_length=SIGLIP2_TEXT_CHUNK_TOKENS,
                           truncation=True, return_tensors="pt").to(device)
        out = siglip_model.get_text_features(**inputs)
        return out.pooler_output if hasattr(out, "pooler_output") else out

    def encode_siglip2_images_batched(image_paths, chunk_size=IMAGE_ENCODE_CHUNK_SIZE):
        """Batched + chunked image encoding. Returns (len(image_paths), VISION_DIM)."""
        if len(image_paths) <= chunk_size:
            images = [Image.open(p).convert("RGB") for p in image_paths]
            inputs = processor(images=images, return_tensors="pt").to(device)
            out = siglip_model.get_image_features(**inputs)
            return out.pooler_output if hasattr(out, "pooler_output") else out
        chunks = []
        for i in range(0, len(image_paths), chunk_size):
            batch_paths = image_paths[i:i + chunk_size]
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            inputs = processor(images=images, return_tensors="pt").to(device)
            out = siglip_model.get_image_features(**inputs)
            chunks.append(out.pooler_output if hasattr(out, "pooler_output") else out)
        return torch.cat(chunks, dim=0)

    class ProjectionHead(nn.Module):
        def __init__(self, in_dim, out_dim=SHARED_DIM):
            super().__init__()
            self.proj = nn.Linear(in_dim, out_dim)
        def forward(self, x):
            return F.normalize(self.proj(x), p=2, dim=-1)

    head_text_siglip2  = ProjectionHead(TEXT_DIM).to(device)
    head_image_siglip2 = ProjectionHead(VISION_DIM).to(device)

    head_params       = list(head_text_siglip2.parameters()) + list(head_image_siglip2.parameters())
    vision_trainable  = [p for p in vision_model.parameters() if p.requires_grad]
    text_trainable    = [p for p in text_model.parameters() if p.requires_grad]

    param_groups = [{"params": head_params, "lr": HEAD_LEARNING_RATE}]
    if vision_trainable:
        param_groups.append({"params": vision_trainable, "lr": SIGLIP2_ENCODER_LR})
    if text_trainable:
        param_groups.append({"params": text_trainable, "lr": SIGLIP2_ENCODER_LR})
    optimizer = torch.optim.AdamW(param_groups)

    logger.info("build_models_imagetext(n=%d): VISION_DIM=%d, TEXT_DIM=%d, "
                "vision trainable=%d, text trainable=%d, optimizer groups=%d%s",
                n_unfrozen_layers, VISION_DIM, TEXT_DIM,
                len(vision_trainable), len(text_trainable), len(param_groups),
                " (frozen baseline: heads only)" if n_unfrozen_layers == 0 else "")

    return {
        "siglip_model": siglip_model,
        "vision_model": vision_model, "text_model": text_model,
        "encode_siglip2_images_batched": encode_siglip2_images_batched,
        "encode_siglip2_text_chunks": encode_siglip2_text_chunks,
        "head_text_siglip2": head_text_siglip2, "head_image_siglip2": head_image_siglip2,
        "optimizer": optimizer,
    }

logger.info("build_models_imagetext(n) ready (mean-pool text reduction).")

2026-10-22 19:39:59,515 : INFO : build_models_imagetext(n) ready (mean-pool text reduction).


## Section 5 — FN-masked contrastive loss (MEAN-POOLED text)

Per-chunk projection → **mean across chunks** → re-normalize → single similarity against image. No `max(dim=...)` anywhere.

In [8]:
def make_batches(records, batch_size=BATCH_SIZE, shuffle=True):
    idx = np.arange(len(records))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start:start + batch_size]
        yield [records[i] for i in batch_idx]


def masked_symmetric_infonce(sim_matrix, mask_matrix, temperature=TEMPERATURE):
    B = sim_matrix.shape[0]
    logits = sim_matrix / temperature
    neg_inf = torch.finfo(logits.dtype).min
    mask_matrix = mask_matrix.clone()
    mask_matrix.fill_diagonal_(False)
    logits = logits.masked_fill(mask_matrix, neg_inf)
    targets = torch.arange(B, device=logits.device)
    loss_i2j = F.cross_entropy(logits, targets)
    loss_j2i = F.cross_entropy(logits.T, targets)
    return (loss_i2j + loss_j2i) / 2


def meanpool_text_chunks(models, text):
    """Encode text into 64-token chunks, project each chunk with head_text_siglip2,
    mean-pool across chunks, re-normalize to unit length. Returns (SHARED_DIM,).

    Mean-pooling (unlike max-similarity) forces EVERY chunk to contribute to the
    text representation, so the model must encode the whole passage, not just
    its most image-like fragment. This matches NB 09's finding.
    """
    chunks_raw = models["encode_siglip2_text_chunks"](text)   # (n_chunks, TEXT_DIM)
    projected  = models["head_text_siglip2"](chunks_raw)      # (n_chunks, SHARED_DIM), each L2-normalized
    pooled     = projected.mean(dim=0)                         # (SHARED_DIM,)
    pooled     = F.normalize(pooled, p=2, dim=0)               # re-normalize after averaging
    return pooled


def compute_imagetext_loss(batch, models):
    texts       = [r["text"] for r in batch]
    image_paths = [r["image_path"] for r in batch]
    B = len(batch)

    image_embs_raw = models["encode_siglip2_images_batched"](image_paths)          # (B, VISION_DIM)
    image_emb = models["head_image_siglip2"](image_embs_raw)                        # (B, SHARED_DIM)

    text_pooled = torch.stack([meanpool_text_chunks(models, t) for t in texts])     # (B, SHARED_DIM)

    sim = image_emb @ text_pooled.T                                                 # (B, B), single similarity

    mask = torch.zeros((B, B), dtype=torch.bool, device=device)
    for i in range(B):
        for j in range(B):
            if i != j and is_masked_image_image(batch[i], batch[j]):
                mask[i, j] = True

    return masked_symmetric_infonce(sim, mask)

logger.info("compute_imagetext_loss() (FN-masked, SigLIP2, mean-pooled text) ready.")

2026-10-22 19:39:59,671 : INFO : compute_imagetext_loss() (FN-masked, SigLIP2, mean-pooled text) ready.


## Section 6 — GT metrics + evaluation function (mean-pooled text)

Same mean-pooling mechanism as training: image batched-chunked, per-sample text mean-pooled, single dot-product similarity. **No per-chunk max anywhere.**

In [9]:
def compute_retrieval_metrics(sim_matrix, k_values=K_VALUES):
    N = sim_matrix.shape[0]
    true_scores = sim_matrix.diag().unsqueeze(1)
    ranks = (sim_matrix >= true_scores).sum(dim=1)
    results = {}
    for k in k_values:
        results[f"R@{k}"] = (ranks <= k).float().mean().item()
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = N
    return results


def evaluate_imagetext(models):
    models["siglip_model"].eval()
    models["head_text_siglip2"].eval(); models["head_image_siglip2"].eval()
    with torch.no_grad():
        gt_image_embs_raw = models["encode_siglip2_images_batched"](
            [r["image_path"] for r in gt_textimage_records]
        )
        gt_image_emb = models["head_image_siglip2"](gt_image_embs_raw)

        gt_text_pooled = torch.stack([
            meanpool_text_chunks(models, r["text"]) for r in gt_textimage_records
        ])

    sim = gt_image_emb @ gt_text_pooled.T

    summary = pd.DataFrame([
        {"direction": "text -> image", **compute_retrieval_metrics(sim.T)},
        {"direction": "image -> text", **compute_retrieval_metrics(sim)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]

    models["siglip_model"].train()
    models["head_text_siglip2"].train(); models["head_image_siglip2"].train()
    return summary

logger.info("evaluate_imagetext() ready (mean-pooled text).")

2026-10-22 19:39:59,842 : INFO : evaluate_imagetext() ready (mean-pooled text).


## Section 7 — Sanity check (run BEFORE the sweep)

Loss + backward on `n = N_LAYERS_TO_TRY[0]` (= `n=0`). Gradient check is `n`-aware.

In [10]:
_sanity_n = N_LAYERS_TO_TRY[0]
_sanity_models = build_models_imagetext(_sanity_n)
_sanity_batch = textimage_records[:BATCH_SIZE]
_loss_check = compute_imagetext_loss(_sanity_batch, _sanity_models)

logger.info("Sanity (n=%d): loss=%.4f (naive ln(%d)=%.4f)",
            _sanity_n, _loss_check.item(), BATCH_SIZE, math.log(BATCH_SIZE))
if not math.isfinite(_loss_check.item()):
    raise RuntimeError("Sanity loss is NaN/Inf -- stop and debug before running the sweep")

_loss_check.backward()

_grad_vision = any(p.grad is not None and p.grad.abs().sum().item() > 0
                   for p in _sanity_models["vision_model"].parameters() if p.requires_grad)
_grad_text = any(p.grad is not None and p.grad.abs().sum().item() > 0
                 for p in _sanity_models["text_model"].parameters() if p.requires_grad)

if _sanity_n == 0:
    logger.info("Sanity (n=0): both SigLIP2 towers FROZEN -- absence of encoder gradients is expected.")
    logger.info("  vision trainable params receive grad: %s (expected False)", _grad_vision)
    logger.info("  text   trainable params receive grad: %s (expected False)", _grad_text)
    _grad_heads = any(p.grad is not None and p.grad.abs().sum().item() > 0
                      for p in _sanity_models["head_text_siglip2"].parameters()) or \
                  any(p.grad is not None and p.grad.abs().sum().item() > 0
                      for p in _sanity_models["head_image_siglip2"].parameters())
    logger.info("  projection heads receive grad: %s (expected True)", _grad_heads)
    if not _grad_heads:
        raise RuntimeError("Sanity failed at n=0: no gradients reaching projection heads either")
else:
    logger.info("Gradients reaching unfrozen vision layers: %s", _grad_vision)
    logger.info("Gradients reaching unfrozen text layers:   %s", _grad_text)
    if not (_grad_vision and _grad_text):
        raise RuntimeError(
            f"Sanity failed at n={_sanity_n}: both towers should be receiving gradients "
            f"(vision={_grad_vision}, text={_grad_text}). Check optimizer param groups."
        )

del _sanity_models, _loss_check, _sanity_batch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()
logger.info("Allocated after sanity cleanup: %.2f GB", torch.cuda.memory_allocated(device) / 1e9)

2026-10-22 19:40:07,007 : INFO : build_models_imagetext(n=0): VISION_DIM=768, TEXT_DIM=768, vision trainable=0, text trainable=0, optimizer groups=1 (frozen baseline: heads only)
2026-10-22 19:40:07,835 : INFO : Sanity (n=0): loss=3.4859 (naive ln(32)=3.4657)
2026-10-22 19:40:07,906 : INFO : Sanity (n=0): both SigLIP2 towers FROZEN -- absence of encoder gradients is expected.
2026-10-22 19:40:07,908 : INFO :   vision trainable params receive grad: False (expected False)
2026-10-22 19:40:07,909 : INFO :   text   trainable params receive grad: False (expected False)
2026-10-22 19:40:07,917 : INFO :   projection heads receive grad: True (expected True)
2026-10-22 19:40:08,372 : INFO : Allocated after sanity cleanup: 0.02 GB


## Section 8 — Train-or-resume for one `n` (GT every 2 epochs, `empty_cache` only per config)

Same as 27C's Section 8, but the loss function now uses `meanpool_text_chunks` internally.

In [11]:
def train_or_resume_for_n_layers(n_unfrozen_layers, output_parent, size, records_pool, batch_size,
                                  build_models_fn, loss_fn, eval_fn, state_dict_keys, loss_key,
                                  target_epochs=TARGET_EPOCHS, gt_every_n_epochs=GT_EVERY_N_EPOCHS):
    size_dir = os.path.join(output_parent, f"last_{n_unfrozen_layers}_layers")
    os.makedirs(size_dir, exist_ok=True)
    ckpt_path = os.path.join(size_dir, "latest_checkpoint.pt")
    gt_results_path = os.path.join(size_dir, "gt_results.csv")
    per_epoch_gt_dir = os.path.join(size_dir, "per_epoch_gt")

    label = f"L{n_unfrozen_layers}"

    if os.path.exists(gt_results_path):
        logger.info("[%s] Already completed (%s exists) -- skipping.", label, gt_results_path)
        try:
            return pd.read_csv(gt_results_path)
        except Exception as e:
            logger.warning("[%s] Failed to read existing %s: %s", label, gt_results_path, e)
            return None

    os.makedirs(per_epoch_gt_dir, exist_ok=True)

    rng = random.Random(RANDOM_SEED + size)
    size_records = list(records_pool) if size >= len(records_pool) else rng.sample(records_pool, size)
    logger.info("[%s] Records: %d (fixed subset, seed=%d)", label, len(size_records), RANDOM_SEED + size)

    models = build_models_fn(n_unfrozen_layers)

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        start_epoch = ckpt["epoch"]
        for key in state_dict_keys:
            models[key].load_state_dict(ckpt[key])
        models["optimizer"].load_state_dict(ckpt["optimizer"])
        global_step = ckpt["global_step"]
        history = ckpt.get("history", {"epoch": []})
        if start_epoch >= target_epochs:
            logger.info("=" * 70)
            logger.info("=== %s: checkpoint at target (%d/%d) -- finalizing without training ===",
                        label, start_epoch, target_epochs)
            logger.info("=" * 70)
        else:
            logger.info("=" * 70)
            logger.info("=== %s: resuming epoch %d -> target %d ===", label, start_epoch, target_epochs)
            logger.info("=" * 70)
        logger.info("[%s] Loaded: epoch %d, step %d, history keys=%s",
                    label, start_epoch, global_step, list(history.keys()))
    else:
        start_epoch = 0
        global_step = 0
        history = {"epoch": []}
        logger.info("=" * 70)
        logger.info("=== %s: NO checkpoint at %s -- training FRESH from epoch 0 -> target %d ===",
                    label, ckpt_path, target_epochs)
        logger.info("=" * 70)

    def save_checkpoint(epoch, global_step, history):
        tmp_path = ckpt_path + ".tmp"
        state = {key: models[key].state_dict() for key in state_dict_keys}
        state["optimizer"] = models["optimizer"].state_dict()
        state["epoch"] = epoch
        state["global_step"] = global_step
        state["history"] = history
        torch.save(state, tmp_path)
        os.replace(tmp_path, ckpt_path)

    np.random.seed(RANDOM_SEED)
    random.seed(RANDOM_SEED)

    training_start_time = time.time()
    stop_early = False

    try:
        for epoch in range(start_epoch, target_epochs):
            epoch_losses = []
            batches = list(make_batches(size_records, batch_size))
            n_steps = len(batches)

            for step in range(n_steps):
                models["optimizer"].zero_grad()
                loss = loss_fn(batches[step], models)
                loss.backward()
                models["optimizer"].step()
                global_step += 1
                epoch_losses.append(loss.item())

                elapsed_hours = (time.time() - training_start_time) / 3600
                if step % LOG_EVERY_N_STEPS == 0:
                    logger.info("[%s] Epoch %d/%d, step %d/%d, loss=%.4f, elapsed=%.2fh",
                                label, epoch + 1, target_epochs, step, n_steps, loss.item(), elapsed_hours)

                if global_step % CHECKPOINT_EVERY_N_STEPS == 0:
                    save_checkpoint(epoch, global_step, history)

                if elapsed_hours > MAX_RUNTIME_HOURS_PER_SIZE:
                    logger.warning("[%s] Runtime limit reached. Saving and stopping.", label)
                    save_checkpoint(epoch, global_step, history)
                    stop_early = True
                    break
            if stop_early:
                break

            mean_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
            history["epoch"].append(epoch + 1)
            history.setdefault(loss_key, []).append(mean_loss)
            logger.info("[%s] Epoch %d done: loss=%.4f", label, epoch + 1, mean_loss)
            save_checkpoint(epoch + 1, global_step, history)

            is_gt_epoch = ((epoch + 1) % gt_every_n_epochs == 0) or ((epoch + 1) == target_epochs)
            if is_gt_epoch:
                epoch_gt_summary = eval_fn(models)
                epoch_gt_summary.insert(0, "train_size", size)
                epoch_gt_summary.insert(1, "epoch", epoch + 1)
                epoch_gt_summary.insert(2, "n_unfrozen_layers", n_unfrozen_layers)
                epoch_gt_path = os.path.join(per_epoch_gt_dir, f"epoch_{epoch + 1:02d}_gt_results.csv")
                epoch_gt_summary.to_csv(epoch_gt_path, index=False)
                logger.info("[%s] Epoch %d GT results:\n%s",
                            label, epoch + 1,
                            epoch_gt_summary.drop(columns=["train_size", "n_unfrozen_layers"]).to_string(index=False))
            else:
                logger.info("[%s] Epoch %d: GT skipped (cadence=%d)", label, epoch + 1, gt_every_n_epochs)

        pd.DataFrame(history).to_csv(os.path.join(size_dir, "training_history.csv"), index=False)
        for key in state_dict_keys:
            fname_stem = _MODEL_FILE_RENAMES.get(key, key)
            torch.save(models[key].state_dict(), os.path.join(size_dir, f"{fname_stem}.pt"))
        logger.info("[%s] Saved updated outputs.", label)

        gt_summary = eval_fn(models)
        gt_summary.insert(0, "train_size", size)
        gt_summary.insert(1, "n_unfrozen_layers", n_unfrozen_layers)
        gt_summary.to_csv(gt_results_path, index=False)
        logger.info("[%s] Final GT results:\n%s", label, gt_summary.to_string(index=False))
        return gt_summary

    except torch.cuda.OutOfMemoryError:
        logger.error("[%s] OOM. Skipping.", label)
        return None
    finally:
        # Clean up once per config (NOT per step)
        try:
            del models
        except NameError:
            pass
        gc.collect()
        if torch.cuda.is_available():
            with torch.cuda.device(CUDA):
                torch.cuda.empty_cache()
                torch.cuda.synchronize()

logger.info("train_or_resume_for_n_layers() ready.")

2026-10-22 19:40:08,396 : INFO : train_or_resume_for_n_layers() ready.


## Section 9 — Sweep over `n`

`n=0` runs first (baseline). Then `n=2, 4, 8, 12`.

In [13]:
results = []
for n in N_LAYERS_TO_TRY:
    logger.info("#" * 70)
    logger.info("### SWEEP: n_unfrozen_layers = %d %s###",
                n, "(frozen baseline) " if n == 0 else "")
    logger.info("#" * 70)
    r = train_or_resume_for_n_layers(
        n_unfrozen_layers=n, output_parent=OUTPUT_PARENT, size=SIZE, records_pool=textimage_records,
        batch_size=BATCH_SIZE, build_models_fn=build_models_imagetext,
        loss_fn=compute_imagetext_loss, eval_fn=evaluate_imagetext,
        state_dict_keys=["vision_model", "text_model", "head_text_siglip2", "head_image_siglip2"],
        loss_key="imagetext_loss",
    )
    if r is not None:
        results.append(r)

logger.info("Sweep complete. %d n values produced results.", len(results))

2026-10-22 21:31:02,572 : INFO : ######################################################################
2026-10-22 21:31:02,575 : INFO : ### SWEEP: n_unfrozen_layers = 0 (frozen baseline) ###
2026-10-22 21:31:02,577 : INFO : ######################################################################
2026-10-22 21:31:02,581 : INFO : [L0] Records: 2500 (fixed subset, seed=2542)
2026-10-22 21:31:08,016 : INFO : build_models_imagetext(n=0): VISION_DIM=768, TEXT_DIM=768, vision trainable=0, text trainable=0, optimizer groups=1 (frozen baseline: heads only)
2026-10-22 21:31:12,367 : INFO : ======================================================================
2026-10-22 21:31:12,369 : INFO : === L0: resuming epoch 20 -> target 30 ===
2026-10-22 21:31:12,370 : INFO : ======================================================================
2026-10-22 21:31:12,371 : INFO : [L0] Loaded: epoch 20, step 1580, history keys=['epoch', 'imagetext_loss']
2026-10-22 21:31:12,900 : INFO : [L0] Epoch 21/30, step

## Section 10 — Combined summaries

- **`summary_all_n.csv`** — final GT per `n` (`train_size, n_unfrozen_layers, source, direction, N, R@1..MRR`). All rows tagged `source=NB27D`.
- **`summary_per_epoch_all_n.csv`** — every `(n, epoch, direction)` row from `per_epoch_gt/` (epochs 2, 4, …, 30).

The embedded historical outputs are tagged `n_unfrozen_layers=16`; with 12-layer towers this means all layers were unfrozen. The corrected source configuration records that setting as `n=12`.

No reference is loaded (no comparable external run with mean-pooling at this schedule).

In [14]:
def collect_summary_final(output_parent):
    rows = []
    for entry in sorted(os.listdir(output_parent)):
        m = re.match(r"last_(\d+)_layers$", entry)
        if not m:
            continue
        p = os.path.join(output_parent, entry, "gt_results.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            df["source"] = "NB27D"
            rows.append(df)
        else:
            logger.warning("No gt_results.csv at %s", p)
    if not rows:
        return None
    first_cols = ["train_size", "n_unfrozen_layers", "source", "direction",
                  "N", "R@1", "R@3", "R@5", "R@10", "MRR"]
    df = pd.concat(rows, ignore_index=True)
    df = df[[c for c in first_cols if c in df.columns]]
    df = (df.sort_values(["n_unfrozen_layers", "source", "direction"])
            .reset_index(drop=True))
    out = os.path.join(output_parent, "summary_all_n.csv")
    df.to_csv(out, index=False)
    logger.info("Final summary written to %s:\n%s", out, df.to_string(index=False))
    return df


def collect_summary_per_epoch(output_parent):
    rows = []
    for entry in sorted(os.listdir(output_parent)):
        m = re.match(r"last_(\d+)_layers$", entry)
        if not m:
            continue
        per_epoch_dir = os.path.join(output_parent, entry, "per_epoch_gt")
        if not os.path.isdir(per_epoch_dir):
            continue
        for f in sorted(os.listdir(per_epoch_dir)):
            if f.startswith("epoch_") and f.endswith("_gt_results.csv"):
                rows.append(pd.read_csv(os.path.join(per_epoch_dir, f)))
    if not rows:
        return None
    df = (pd.concat(rows, ignore_index=True)
            .drop_duplicates(subset=["n_unfrozen_layers", "epoch", "direction"], keep="last")
            .sort_values(["n_unfrozen_layers", "epoch", "direction"])
            .reset_index(drop=True))
    out = os.path.join(output_parent, "summary_per_epoch_all_n.csv")
    df.to_csv(out, index=False)
    logger.info("Per-epoch summary written to %s (%d rows)", out, len(df))
    return df


summary_all_n = collect_summary_final(OUTPUT_PARENT)
summary_per_epoch_all_n = collect_summary_per_epoch(OUTPUT_PARENT)

try:
    display(summary_all_n)
except NameError:
    print(summary_all_n)

if summary_per_epoch_all_n is not None:
    last_epoch = summary_per_epoch_all_n["epoch"].max()
    compact = summary_per_epoch_all_n[summary_per_epoch_all_n["epoch"] == last_epoch]
    try:
        display(compact)
    except NameError:
        print(compact)

2026-10-22 22:09:49,611 : INFO : Final summary written to /projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/textimage_siglip2_meanpool/summary_all_n.csv:
 train_size  n_unfrozen_layers source     direction   N   R@1   R@3   R@5  R@10      MRR
       2500                  0  NB27D image -> text 500 0.092 0.174 0.220 0.288 0.162062
       2500                  0  NB27D text -> image 500 0.082 0.168 0.200 0.282 0.154035
       2500                  2  NB27D image -> text 500 0.102 0.194 0.256 0.326 0.178261
       2500                  2  NB27D text -> image 500 0.096 0.190 0.258 0.330 0.172073
       2500                  4  NB27D image -> text 500 0.096 0.192 0.246 0.314 0.174227
       2500                  4  NB27D text -> image 500 0.106 0.208 0.240 0.320 0.179826
       2500                  8  NB27D image -> text 500 0.092 0.180 0.212 0.284 0.163062
       2500                  8  NB27D text -> image 500 0.102 0.178 0.218 0.286 0.168577
       2500                 16  N

,train_size,n_unfrozen_layers,source,direction,N,R@1,R@3,R@5,R@10,MRR
0,2500,0,NB27D,image -> text,500,0.092,0.174,0.220,0.288,0.162062
1,2500,0,NB27D,text -> image,500,0.082,0.168,0.200,0.282,0.154035
2,2500,2,NB27D,image -> text,500,0.102,0.194,0.256,0.326,0.178261
3,2500,2,NB27D,text -> image,500,0.096,0.190,0.258,0.330,0.172073
4,2500,4,NB27D,image -> text,500,0.096,0.192,0.246,0.314,0.174227
5,2500,4,NB27D,text -> image,500,0.106,0.208,0.240,0.320,0.179826
6,2500,8,NB27D,image -> text,500,0.092,0.180,0.212,0.284,0.163062
7,2500,8,NB27D,text -> image,500,0.102,0.178,0.218,0.286,0.168577
8,2500,16,NB27D,image -> text,500,0.096,0.182,0.224,0.310,0.166082
9,2500,16,NB27D,text -> image,500,0.104,0.192,0.242,0.310,0.175960


,train_size,epoch,n_unfrozen_layers,direction,N,R@1,R@3,R@5,R@10,MRR
28,2500,30,0,image -> text,500,0.092,0.174,0.220,0.288,0.162062
29,2500,30,0,text -> image,500,0.082,0.168,0.200,0.282,0.154035
58,2500,30,2,image -> text,500,0.102,0.194,0.256,0.326,0.178261
59,2500,30,2,text -> image,500,0.096,0.190,0.258,0.330,0.172073
88,2500,30,4,image -> text,500,0.096,0.192,0.246,0.314,0.174227
89,2500,30,4,text -> image,500,0.106,0.208,0.240,0.320,0.179826
118,2500,30,8,image -> text,500,0.092,0.180,0.212,0.284,0.163062
119,2500,30,8,text -> image,500,0.102,0.178,0.218,0.286,0.168577
148,2500,30,16,image -> text,500,0.096,0.182,0.224,0.310,0.166082
149,2500,30,16,text -> image,500,0.104,0.192,0.242,0.310,0.175960
